# Libraries

In [223]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

project_root = current_dir.parents[2]

csv_path_subject = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Participant_Status_02Mar2026.csv"
csv_path_demo = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Demographics_02Mar2026.csv"
csv_path_econ = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Socio-Economics_02Mar2026.csv"
csv_path_age = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Age_at_visit_02Mar2026.csv"
csv_path_vital = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Vital_Signs_02Mar2026.csv"


# Subject Data

In [224]:
subject_data=pd.read_csv(csv_path_subject)
subject_data=subject_data.loc[subject_data['COHORT_DEFINITION']=="Parkinson's Disease"]
patnos=subject_data['PATNO'].unique().tolist()
print("Subject data:",subject_data.shape)
cols_imp=['PATNO','ENRLLRRK2','ENRLGBA']
subject_data=subject_data[cols_imp].copy()
subject_data.head() # no hay nan

Subject data: (2062, 30)


,PATNO,ENRLLRRK2,ENRLGBA
1,3001,0,0
2,3002,0,0
3,3003,0,0
5,3005,0,0
6,3006,0,0


# Demographic Data

In [225]:
demo_data=pd.read_csv(csv_path_demo)
demo_data=demo_data.loc[demo_data['PATNO'].isin(patnos)]
print("Demographics data:",demo_data.shape)
demo_data=demo_data[['PATNO','SEX','RAWHITE']].copy() # no hay nan

Demographics data: (2050, 29)


# Socio-Economics Data

In [226]:
socio_data=pd.read_csv(csv_path_econ)
socio_data=socio_data.loc[(socio_data['PATNO'].isin(patnos))&(socio_data['EVENT_ID'].isin(['BL','SC']))]
print("Socio-Economics data:",socio_data.shape)
socio_data=socio_data[['PATNO','EDUCYRS']].copy()
socio_data.fillna(socio_data.mean(numeric_only=True), inplace=True)
socio_data.head() # no hay nan

Socio-Economics data: (1997, 11)


,PATNO,EDUCYRS
2,3001,16.0
4,3002,16.0
6,3003,16.0
10,3005,16.0
11,3006,14.0


# Subject/Demo/SocioEcon

In [227]:
# Merge datasets
data = pd.merge(subject_data, demo_data, on='PATNO', how='inner')
data = pd.merge(data, socio_data, on='PATNO', how='inner')
print("Merged data:", data.shape)
data.head()

Merged data: (1997, 6)


,PATNO,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS
0,3001,0,0,1.0,1.0,16.0
1,3002,0,0,0.0,1.0,16.0
2,3003,0,0,0.0,1.0,16.0
3,3005,0,0,0.0,1.0,16.0
4,3006,0,0,0.0,1.0,14.0


# Age at Visit + Data Total

In [228]:
age_data=pd.read_csv(csv_path_age)
age_data=age_data.loc[age_data['PATNO'].isin(patnos)&(age_data['EVENT_ID'].isin(['BL','SC','V04','V06','V08','V10','V12']))]

# 1. si hay patnos que tienen BL y SC eliminamos los SC
# Identificar PATNOs que tienen tanto BL como SC
patnos_bl = set(age_data.loc[age_data['EVENT_ID'] == 'BL', 'PATNO'])
patnos_sc = set(age_data.loc[age_data['EVENT_ID'] == 'SC', 'PATNO'])

patnos_con_bl_y_sc = patnos_bl.intersection(patnos_sc)

# Eliminar los registros SC solo para esos PATNOs
age_data = age_data[
    ~((age_data['PATNO'].isin(patnos_con_bl_y_sc)) & (age_data['EVENT_ID'] == 'SC'))
]

age_data['EVENT_ID']= age_data['EVENT_ID'].replace({'SC':'BL'})
# corregir inconsistencias en los datos de edad
# sabemos que el orden es BL, V04, V06, V08, V10, V12 y que cada visita hay un año de diferencia 
# si hay un PATNO que tiene por ejemplo BL y V08 pero no tiene V04 ni V06, debemos generar los datos de edad para V04 y V06 los cuales son BL+1 y BL+2 respectivamente
AGE_COL = 'AGE_AT_VISIT'  

visit_offsets = {'BL': 0, 'V04': 1, 'V06': 2, 'V08': 3, 'V10': 4, 'V12': 5}
ordered_visits = ['BL', 'V04', 'V06', 'V08', 'V10', 'V12']

# Baseline age por PATNO (ancla)
bl_age = age_data.loc[age_data['EVENT_ID'] == 'BL', ['PATNO', 'AGE_AT_VISIT']].set_index('PATNO')['AGE_AT_VISIT']


# Visitas existentes por PATNO
existing = age_data.loc[age_data['EVENT_ID'].isin(ordered_visits), ['PATNO', 'EVENT_ID']].drop_duplicates()

# Máximo offset observado por PATNO (hasta dónde rellenar)
max_offset = (
    existing.assign(offset=existing['EVENT_ID'].map(visit_offsets))
            .groupby('PATNO')['offset']
            .max()
)

rows_to_add = []
for patno, max_off in max_offset.items():
    if patno not in bl_age.index:
        continue  # sin BL no se puede imputar

    present_visits = set(existing.loc[existing['PATNO'] == patno, 'EVENT_ID'])

    for v in ordered_visits:
        off = visit_offsets[v]
        if off <= max_off and v not in present_visits:
            rows_to_add.append({
                'PATNO': patno,
                'EVENT_ID': v,
                AGE_COL: float(bl_age.loc[patno]) + off
            })

missing_df = pd.DataFrame(rows_to_add)

# Concatenar: esto NO modifica filas existentes
age_data = pd.concat([age_data, missing_df], ignore_index=True)
age_data = age_data.sort_values(['PATNO', 'EVENT_ID']).reset_index(drop=True)


data=pd.merge(data, age_data, on='PATNO', how='inner')
print("Merged data with age:", data.shape)
data.head()

Merged data with age: (6670, 8)


,PATNO,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,EVENT_ID,AGE_AT_VISIT
0,3001,0,0,1.0,1.0,16.0,BL,65.1
1,3001,0,0,1.0,1.0,16.0,V04,66.2
2,3001,0,0,1.0,1.0,16.0,V06,67.3
3,3001,0,0,1.0,1.0,16.0,V08,68.3
4,3001,0,0,1.0,1.0,16.0,V10,69.2


# Final DF Result

In [229]:
print("Final merged data:", data.shape)
print('Final Cols:', data.columns.tolist())
print("Missing values:\n", data.isna().sum().sum())
csv_path_data = project_root / "DATA_PPMI" / "Results" / "SUBJECT_CHARACTERISTICS.csv"
data.to_csv(csv_path_data, index=False)
data.head()


Final merged data: (6670, 8)
Final Cols: ['PATNO', 'ENRLLRRK2', 'ENRLGBA', 'SEX', 'RAWHITE', 'EDUCYRS', 'EVENT_ID', 'AGE_AT_VISIT']
Missing values:
 0


,PATNO,ENRLLRRK2,ENRLGBA,SEX,RAWHITE,EDUCYRS,EVENT_ID,AGE_AT_VISIT
0,3001,0,0,1.0,1.0,16.0,BL,65.1
1,3001,0,0,1.0,1.0,16.0,V04,66.2
2,3001,0,0,1.0,1.0,16.0,V06,67.3
3,3001,0,0,1.0,1.0,16.0,V08,68.3
4,3001,0,0,1.0,1.0,16.0,V10,69.2
